Go through four features: coordinates of heavy atoms, distances between heavy atoms, angle between bonds, and dihedral angles.
Coor is not roto-translational invariant, distance has some bug to do TICA, angle change is so fast so that no useful info is providing
Dihedral is a good feature to go.

In [ ]:
import pyemma
from pyemma.util.contexts import settings
import pyemma.msm as msm
import pyemma.plots as mplt
import mdtraj as md
import pickle
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append("../")
from descriptors import *

In [ ]:
def get_torsions_in_traj(traj, mol):
    torsion_ring = get_torsions_idx_mol(mol)[1]
    torsion_non_ring = get_torsions_idx_mol(mol)[0]
    torsion_all = torsion_ring + torsion_non_ring
    torsions = md.compute_dihedrals(traj, torsion_all)  # shape (n_frames, n_torsions)
    return torsions

In [ ]:
molecule_prefix = '1_results\\C_C_H_1C_C_H_CCCO_C1_'
molecule_prefix = '0_results\\CC_C_C_H_C_C_H_O_CO_'
xtc_path = f'eval/data_sample/{molecule_prefix}15\\traj.xtc'
pdb_path = f'eval/data_sample/{molecule_prefix}15\\system.pdb'
mol_path = f'eval/data_sample/{molecule_prefix}15\\mol.pkl'

trajs = [f"eval/data_sample/{molecule_prefix}{i}\\traj.xtc" for i in range(15,20)]
traj = md.load(xtc_path, top=pdb_path)
mol = pickle.load(open(mol_path, 'rb'))

In [ ]:
# get torsion atom indices
torsion_index = np.array(get_torsions_idx_mol(mol)[1]+get_torsions_idx_mol(mol)[0]).reshape(-1,4)
print(torsion_index)
# get bond angle indices
angle_index = np.array(get_bond_angles_idx_mol(mol)).reshape(-1,3)
# print(angle_index)

In [ ]:
labels = ["dihedral", "angle", "distance", "coordinate"]
# Feature: dihedral angle
feat_dihedral = pyemma.coordinates.featurizer(pdb_path)
feat_dihedral.add_dihedrals(torsion_index, periodic=False)
print(feat_dihedral.dimension())

# Feature: angle between heavy atoms
feat_angle = pyemma.coordinates.featurizer(pdb_path)
feat_angle.add_angles(angle_index, periodic=False)
print(feat_angle.dimension())

# Feature: Distance between heavy atoms
feat_distance = pyemma.coordinates.featurizer(pdb_path)
heavy_atom_distance_pairs = feat_distance.pairs(feat_distance.select_Heavy())
feat_distance.add_distances(heavy_atom_distance_pairs, periodic=False)
print(feat_distance.dimension())

# Feature: coordinate of heavy atoms, this feature needs alignment of traj
feat_coor = pyemma.coordinates.featurizer(pdb_path)
feat_coor.add_selection(feat_coor.select_Heavy())
print(feat_coor.dimension())

feat_combined = pyemma.coordinates.featurizer(pdb_path)
feat_combined.add_dihedrals(torsion_index, periodic=False)
feat_combined.add_distances(heavy_atom_distance_pairs, periodic=False)
# feat_combined.add_selection(feat_combined.select_Heavy())
print(feat_combined.dimension())

In [ ]:
dihedral_data = pyemma.coordinates.load(trajs, features=feat_dihedral)
angle_data = pyemma.coordinates.load(trajs, features=feat_angle)
distance_data = pyemma.coordinates.load(trajs, features=feat_distance)
coor_data = pyemma.coordinates.load(trajs, features=feat_coor)
combined_data = pyemma.coordinates.load(trajs, features=feat_combined)

In [ ]:
# Plot distribution of values in one feature
fig, ax = plt.subplots(figsize=(6, 4))
pyemma.plots.plot_feature_histograms(np.concatenate(dihedral_data), feature_labels=feat_dihedral, ax=ax)
fig.tight_layout()

In [ ]:
# It's good to scan a wide range of lag times to find the best lag time.
# We want small number of dimensions can explain most of the kinetic variance.
# We should choose the lag time when we see elbow point in that trace.
# QM9 has very few dihedral angles, this might be helpful for drug
lags = [1,10, 100, 1000, 3000]
dims = [i + 1 for i in range(feat_dihedral.dimension())]

fig, ax = plt.subplots()
for i, lag in enumerate(lags):
    scores_ = np.array([score_cv(dihedral_data, dim, lag)
                        for dim in dims])
    scores = np.mean(scores_, axis=1)
    errors = np.std(scores_, axis=1, ddof=1)
    color = 'C{}'.format(i)
    ax.fill_between(dims, scores - errors, scores + errors, alpha=0.3, facecolor=color)
    ax.plot(dims, scores, '--o', color=color, label='lag={:.1f}ps'.format(lag * 0.4))
ax.legend()
ax.set_xlabel('number of dimensions')
ax.set_ylabel('VAMP2 score')
fig.tight_layout()

In [ ]:
########## TICA on dihedral angles ##########
# Please play with the lag time and see how it affects the TICA output.
# Actually, the result does not change much...
lag = 10
tica = pyemma.coordinates.tica(dihedral_data, lag=lag)
tica_output = tica.get_output()
tica_concatenated = np.concatenate(tica_output)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
pyemma.plots.plot_feature_histograms(
    tica_concatenated,
    ['IC {}'.format(i + 1) for i in range(tica.dimension())],
    ax=axes[0])
axes[0].set_title('lag time = {} steps'.format(lag))
axes[1].set_title(
    'Density, actual dimension = {}'.format(tica.dimension()))
pyemma.plots.plot_density(
    *tica_concatenated[:, :2].T, ax=axes[1], cbar=False)
pyemma.plots.plot_free_energy(
    *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
for ax in axes[1:].flat:
    ax.set_xlabel('IC 1')
    ax.set_ylabel('IC 2')
axes[2].set_title('Pseudo free energy')
fig.tight_layout()

Stuffs below are code used for selecting features (Dihedral, angle, coordinates, distance...)

In [ ]:
### Only works for specific lag time
dim = 2
lag = 25
# Distance beween heavy atoms
score_pair_heavy_atom_dists = pyemma.coordinates.vamp(distance_data[:-1], dim=dim, lag=lag).score(
    test_data=distance_data[-1],
    score_method='VAMP2')
print('VAMP2-score: {:.2f}'.format(score_pair_heavy_atom_dists))
# Coordinate of heavy atoms
score_pair_heavy_atom_dists = pyemma.coordinates.vamp(coor_data[:-1], dim=dim, lag=lag).score(
    test_data=coor_data[-1],
    score_method='VAMP2')
print('VAMP2-score: {:.2f}'.format(score_pair_heavy_atom_dists))
# Dihedral angle
score_pair_heavy_atom_dists = pyemma.coordinates.vamp(dihedral_data[:-1], dim=dim, lag=lag).score(
    test_data=dihedral_data[-1],
    score_method='VAMP2')
print('VAMP2-score: {:.2f}'.format(score_pair_heavy_atom_dists))
# Angle
score_pair_heavy_atom_dists = pyemma.coordinates.vamp(angle_data[:-1], dim=dim, lag=lag).score(
    test_data=angle_data[-1],
    score_method='VAMP2')
print('VAMP2-score: {:.2f}'.format(score_pair_heavy_atom_dists))

In [ ]:
def score_cv(data, dim, lag, number_of_splits=5, validation_fraction=0.5):
    """Compute a cross-validated VAMP2 score.

    We randomly split the list of independent trajectories into
    a training and a validation set, compute the VAMP2 score,
    and repeat this process several times.

    Parameters
    ----------
    data : list of numpy.ndarrays
        The input data.
    dim : int
        Number of processes to score; equivalent to the dimension
        after projecting the data with VAMP2.
    lag : int
        Lag time for the VAMP2 scoring.
    number_of_splits : int, optional, default=10
        How often do we repeat the splitting and score calculation.
    validation_fraction : int, optional, default=0.5
        Fraction of trajectories which should go into the validation
        set during a split.
    """
    # we temporarily suppress very short-lived progress bars
    with pyemma.util.contexts.settings(show_progress_bars=False):
        nval = int(len(data) * validation_fraction)
        scores = np.zeros(number_of_splits)
        for n in range(number_of_splits):
            ival = np.random.choice(len(data), size=nval, replace=False)
            vamp = pyemma.coordinates.vamp(
                [d for i, d in enumerate(data) if i not in ival], lag=lag, dim=dim)
            scores[n] = vamp.score([d for i, d in enumerate(data) if i in ival])
    return scores

dim = 2
print(dim)
# To find the best feature we use, we scan a wide range of lag times, choose the feature with the highest score.
fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, lag in zip(axes.flat, [10, 100, 1000]):
    torsions_scores = score_cv(dihedral_data, lag=lag, dim=dim)
    scores = [torsions_scores.mean()]
    errors = [torsions_scores.std()]
    positions_scores = score_cv(angle_data, lag=lag, dim=dim)
    scores += [positions_scores.mean()]
    errors += [positions_scores.std()]
    distances_scores = score_cv(distance_data, lag=lag, dim=dim)
    scores += [distances_scores.mean()]
    errors += [distances_scores.std()]
    coordinate_scores = score_cv(coor_data, lag=lag, dim=dim)
    scores += [coordinate_scores.mean()]
    errors += [coordinate_scores.std()]

    ax.bar(labels, scores, yerr=errors, color=['C0','C1', 'C2','C3'])
    ax.set_title(r'lag time $\tau$={:.1f}ps'.format(lag * 0.4))
    if lag == 5:
        # save for later
        vamp_bars_plot = dict(
            labels=labels, scores=scores, errors=errors, dim=dim, lag=lag)
axes[0].set_ylabel('VAMP2 score')
fig.tight_layout()
plt.show()

In [ ]:
# # Plot distribution of values in one feature
# fig, ax = plt.subplots(figsize=(10, 7))
# pyemma.plots.plot_feature_histograms(np.concatenate(combined_data), feature_labels=feat_combined, ax=ax)
# fig.tight_layout()

In [ ]:
# It's good to scan a wide range of lag times to find the best lag time.
# We want small number of dimensions can explain most of the kinetic.
# We should choose the lag time when we see elbow point in that trace.
lags = [10, 100, 300, 1000, 3000]
dims = [i + 1 for i in range(20)]

fig, ax = plt.subplots()
for i, lag in enumerate(lags):
    scores_ = np.array([score_cv(coor_data, dim, lag)
                        for dim in dims])
    scores = np.mean(scores_, axis=1)
    errors = np.std(scores_, axis=1, ddof=1)
    color = 'C{}'.format(i)
    ax.fill_between(dims, scores - errors, scores + errors, alpha=0.3, facecolor=color)
    ax.plot(dims, scores, '--o', color=color, label='lag={:.1f}ps'.format(lag * 0.4))
ax.legend()
ax.set_xlabel('number of dimensions')
ax.set_ylabel('VAMP2 score')
fig.tight_layout()

In [ ]:
# The same, but for dihedral feature.
# QM9 has very few dihedral angles.
lags = [1,10, 100, 1000, 3000]
dims = [i + 1 for i in range(4)]

fig, ax = plt.subplots()
for i, lag in enumerate(lags):
    scores_ = np.array([score_cv(dihedral_data, dim, lag)
                        for dim in dims])
    scores = np.mean(scores_, axis=1)
    errors = np.std(scores_, axis=1, ddof=1)
    color = 'C{}'.format(i)
    ax.fill_between(dims, scores - errors, scores + errors, alpha=0.3, facecolor=color)
    ax.plot(dims, scores, '--o', color=color, label='lag={:.1f}ps'.format(lag * 0.4))
ax.legend()
ax.set_xlabel('number of dimensions')
ax.set_ylabel('VAMP2 score')
fig.tight_layout()

In [ ]:
########## TICA on dihedral angles ##########
# Please play with the lag time and see how it affects the TICA output.
# Actually, the result does not change much...
lag = 10
tica = pyemma.coordinates.tica(dihedral_data, lag=lag)
tica_output = tica.get_output()
tica_concatenated = np.concatenate(tica_output)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
pyemma.plots.plot_feature_histograms(
    tica_concatenated,
    ['IC {}'.format(i + 1) for i in range(tica.dimension())],
    ax=axes[0])
axes[0].set_title('lag time = {} steps'.format(lag))
axes[1].set_title(
    'Density, actual dimension = {}'.format(tica.dimension()))
pyemma.plots.plot_density(
    *tica_concatenated[:, :2].T, ax=axes[1], cbar=False)
pyemma.plots.plot_free_energy(
    *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
for ax in axes[1:].flat:
    ax.set_xlabel('IC 1')
    ax.set_ylabel('IC 2')
axes[2].set_title('Pseudo free energy')
fig.tight_layout()

In [ ]:
########### TICA on coor ##########
# Coordinate of heavy atoms is slower than dihedral.
lag = 10
tica = pyemma.coordinates.tica(coor_data, lag=lag)
tica_output = tica.get_output()
tica_concatenated = np.concatenate(tica_output)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
pyemma.plots.plot_feature_histograms(
    tica_concatenated,
    ['IC {}'.format(i + 1) for i in range(tica.dimension())],
    ax=axes[0])
axes[0].set_title('lag time = {} steps'.format(lag))
axes[1].set_title(
    'Density, actual dimension = {}'.format(tica.dimension()))
pyemma.plots.plot_density(
    *tica_concatenated[:, :2].T, ax=axes[1], cbar=False)
pyemma.plots.plot_free_energy(
    *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
for ax in axes[1:].flat:
    ax.set_xlabel('IC 1')
    ax.set_ylabel('IC 2')
axes[2].set_title('Pseudo free energy')
fig.tight_layout()

In [ ]:
########### TICA on distance ##########
# It has bug SOMETIMES
lag = 10
tica = pyemma.coordinates.tica(distance_data, lag=lag)
tica_output = tica.get_output()
tica_concatenated = np.concatenate(tica_output)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
pyemma.plots.plot_feature_histograms(
    tica_concatenated,
    ['IC {}'.format(i + 1) for i in range(tica.dimension())],
    ax=axes[0])
axes[0].set_title('lag time = {} steps'.format(lag))
axes[1].set_title(
    'Density, actual dimension = {}'.format(tica.dimension()))
pyemma.plots.plot_density(
    *tica_concatenated[:, :2].T, ax=axes[1], cbar=False)
pyemma.plots.plot_free_energy(
    *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
for ax in axes[1:].flat:
    ax.set_xlabel('IC 1')
    ax.set_ylabel('IC 2')
axes[2].set_title('Pseudo free energy')
fig.tight_layout()

In [ ]:
########### TICA on combined ##########

lag = 100
tica = pyemma.coordinates.tica(combined_data, lag=lag)
tica_output = tica.get_output()
tica_concatenated = np.concatenate(tica_output)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
pyemma.plots.plot_feature_histograms(
    tica_concatenated,
    ['IC {}'.format(i + 1) for i in range(tica.dimension())],
    ax=axes[0])
axes[0].set_title('lag time = {} steps'.format(lag))
axes[1].set_title(
    'Density, actual dimension = {}'.format(tica.dimension()))
pyemma.plots.plot_density(
    *tica_concatenated[:, :2].T, ax=axes[1], cbar=False)
pyemma.plots.plot_free_energy(
    *tica_concatenated[:, :2].T, ax=axes[2], legacy=False)
for ax in axes[1:].flat:
    ax.set_xlabel('IC 1')
    ax.set_ylabel('IC 2')
axes[2].set_title('Pseudo free energy')
fig.tight_layout()